## Introduction

This notebook covers three powerful areas of Python that are essential for AI/ML development. By the end of this notebook, you should be comfortable with:

* **OOP Deep Dive**: Dunder (magic) methods, class vs instance variables, inheritance, `@property`, `@staticmethod`, `@classmethod`
* **Functional Python**: `map`, `filter`, `reduce`, decorators, generators, `*args`/`**kwargs`
* **NumPy**: Array creation, indexing, slicing, broadcasting, and array math

Please make sure to run <span style="color: red;">all cells</span> regardless of your experience level. Each section builds on the previous one, so read the explanations carefully before attempting the tasks.


## Section 1: Object Oriented Programming (OOP) — Deep Dive

### 1.1 Dunder (Magic) Methods

You already know `__init__`. Python classes support many more **special methods** (also called *dunder methods* because they have **d**ouble **under**scores on both sides). These let your objects behave like built-in Python types — you can make them printable, addable, comparable, and more.

Here is a `Vector` class that demonstrates the most commonly used dunder methods:

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        # Called by print() — meant for end users
        return f"Vector({self.x}, {self.y})"

    def __repr__(self):
        # Called in the shell/debugger — meant for developers
        return f"Vector(x={self.x}, y={self.y})"

    def __len__(self):
        # Called by len() — a 2D vector always has 2 components
        return 2

    def __add__(self, other):
        # Called when you write v1 + v2
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, scalar):
        # Called when you write v * 3
        return Vector(self.x * scalar, self.y * scalar)

    def __eq__(self, other):
        # Called when you write v1 == v2
        return self.x == other.x and self.y == other.y


v1 = Vector(1, 2)
v2 = Vector(3, 4)
print(v1)           # Prints "Vector(1, 2)"
print(repr(v1))     # Prints "Vector(x=1, y=2)"
print(len(v1))      # Prints "2"
print(v1 + v2)      # Prints "Vector(4, 6)"
print(v1 * 3)       # Prints "Vector(3, 6)"
print(v1 == v2)     # Prints "False"


Vector(1, 2)
Vector(x=1, y=2)
2
Vector(4, 6)
Vector(3, 6)
False


### 1.2 Class Variables vs Instance Variables

- **Instance variables** (`self.x`) belong to each individual object — every object has its own copy.
- **Class variables** (defined directly in the class body) are shared across **all** instances of the class.

This distinction matters a lot in ML when you want to track something globally across all objects (e.g., number of models trained).

In [ ]:
class Student:
    school = "NIAI"         # class variable — shared by ALL students

    def __init__(self, name, grade):
        self.name = name    # instance variable — unique to each student
        self.grade = grade  # instance variable — unique to each student

s1 = Student("Ali", "A")
s2 = Student("Sara", "B")

print(s1.school)    # Prints "NIAI"
print(s2.school)    # Prints "NIAI"

# Changing the class variable affects ALL instances
Student.school = "NETSOL Institute of AI"
print(s1.school)    # Prints "NETSOL Institute of AI"
print(s2.school)    # Prints "NETSOL Institute of AI"

# But changing it on one instance only affects that instance
s1.school = "FCC"
print(s1.school)    # Prints "FCC"   (instance variable shadows class variable)
print(s2.school)    # Prints "NETSOL Institute of AI"  (still uses class variable)


NIAI
NIAI
NETSOL Institute of AI
NETSOL Institute of AI
FCC
NETSOL Institute of AI


### 1.3 @property, @staticmethod, @classmethod

Python provides three special decorators for class methods:

| Decorator | Receives | Use case |
|---|---|---|
| `@property` | `self` | Make a method look like an attribute; add validation |
| `@staticmethod` | nothing | Utility function that belongs logically to the class |
| `@classmethod` | `cls` (the class) | Factory methods — alternative ways to create instances |


In [ ]:
class Circle:
    pi = 3.14159    # class variable

    def __init__(self, radius):
        self._radius = radius   # _ prefix = "private by convention"

    @property
    def radius(self):
        return self._radius     # access as c.radius, not c.radius()

    @radius.setter
    def radius(self, value):
        if value < 0:
            raise ValueError("Radius cannot be negative")
        self._radius = value    # validates input before setting

    @property
    def area(self):
        return Circle.pi * self._radius ** 2   # computed on the fly

    @property
    def circumference(self):
        return 2 * Circle.pi * self._radius

    @staticmethod
    def description():
        # No access to self or cls — pure utility
        return "A circle is defined by its radius."

    @classmethod
    def unit_circle(cls):
        # Factory method — creates a Circle with radius = 1
        return cls(1)


c = Circle(5)
print(c.radius)           # Prints "5"
print(c.area)             # Prints "78.53975"
print(c.circumference)    # Prints "31.4159"
print(Circle.description())      # Prints "A circle is defined by its radius."

unit = Circle.unit_circle()
print(unit.radius)        # Prints "1"
print(unit.area)          # Prints "3.14159"

# Setter with validation
c.radius = 10
print(c.radius)           # Prints "10"
# c.radius = -1           # Would raise ValueError


5
78.53975
31.4159
A circle is defined by its radius.
1
3.14159
10


### 1.4 Inheritance and Method Overriding

**Inheritance** lets one class reuse the code of another. The child class gets all the parent's methods and can:
- **Override** them (replace with its own version)
- **Extend** them (call `super()` and add behaviour)

This is the basis of **polymorphism** — you can write code that works on any `Animal` without knowing if it's a `Dog` or `Cat`.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name} makes a sound."

    def __str__(self):
        return f"Animal: {self.name}"


class Dog(Animal):
    def speak(self):              # overrides parent method
        return f"{self.name} says Woof!"


class Cat(Animal):
    def speak(self):
        return f"{self.name} says Meow!"


class GuideDog(Dog):              # multi-level: GuideDog -> Dog -> Animal
    def __init__(self, name, owner):
        super().__init__(name)    # calls Dog.__init__ -> Animal.__init__
        self.owner = owner

    def __str__(self):
        return f"GuideDog: {self.name}, Owner: {self.owner}"


# Polymorphism: same code works for all Animal types
animals = [Dog("Rex"), Cat("Whiskers"), GuideDog("Buddy", "Ahmad")]
for animal in animals:
    print(animal.speak())

print()
print(animals[2])     # Uses GuideDog's __str__


Rex says Woof!
Whiskers says Meow!
Buddy says Woof!

GuideDog: Buddy, Owner: Ahmad


---
## OOP Tasks

### Q1

Create a class `BankAccount` with the following:

- **Instance variables**: `owner` (str), `balance` (float)
- `deposit(amount)` method — adds to balance
- `withdraw(amount)` method — subtracts from balance; raises `ValueError` if insufficient funds
- `__str__` — returns `"Account[owner] | Balance: X"`
- A `@property` called `is_rich` that returns `True` if balance > 100,000
- A `@classmethod` called `zero_account(cls, owner)` that creates an account with balance = 0


In [ ]:
class BankAccount:
    ### Code here
    def __init__(self,owner,balance):
      self.balance=balance
      self.owner=owner

    def deposit(self,amount):
      self.balance+=amount
      return "balance: ",self.balance
    def  withdraw(self,amount):
      if amount>(self.balance):
        raise ValueError(f"withdraw Amount ({amount} is greater than balance {self.balance})")
      self.balance-=amount
    def __str__(self):
     return f"Account[{self.owner}] | Balance: {self.balance}"
    def is_rich(self):
    #  if self.balance>100000:
    #   return True
     return self.balance>100000
    @classmethod
    def  zero_account(cls, owner):

        newAccount=BankAccount(owner,0)
        print(f'Acoount Created')
        # cls.__str__()
        return newAccount


acc = BankAccount("Ali", 50000)
acc.deposit(10000)
print(acc)                # Account[Ali] | Balance: 60000
# print(acc.is_rich)  #  Prints method object
print(acc.is_rich())        # False #Calls the method
acc.deposit(50000)
print(acc.is_rich())        # True

empty = BankAccount.zero_account("Sara")
print(empty)              # Account[Sara] | Balance: 0

try:
    acc.withdraw(200000)  # Should raise ValueError
except ValueError as e:
    print(e)


Account[Ali] | Balance: 60000
False
True
Acoount Created
Account[Sara] | Balance: 0
withdraw Amount (200000 is greater than balance 110000)


### Q2

Create a class `Matrix` that:

- Stores a 2D list in `__init__(self, data)`
- `__str__` prints it row by row (one row per line)
- `__add__` adds two matrices element-wise — raise `ValueError` if shapes don't match
- `__mul__` supports scalar multiplication (e.g., `m * 3`)
- `shape` property that returns `(rows, cols)`
- `@staticmethod` called `identity(n)` that returns an n×n identity matrix as a `Matrix` object


In [ ]:
class Matrix:
    ### Code here
    def __init__(self,data):
      self.data=data
    def __str__(self):
      """Print matrix row by row"""
      result = []
      for row in self.data:
            # Convert each element to string and join with spaces
            result.append(" ".join(str(element) for element in row))
      return "\n".join(result)

    def __add__(self,other):
      newMatrix=[[0 for i in range(len(self.data))]for j in range(len(self.data[0]))]
      if self.shape()==other.shape():
        for i in range(len(self.data)):
          for j in range(len(self.data[0])):
            newMatrix[i][j]=self.data[i][j]+other.data[i][j]
        return newMatrix
      # newMatrix=[[0 for i in range(len(self.data))]for j in range(len(self.data[0]))]
      # newMatrix=zip(self.data,other.data)
      # print(newMatrix)

    def __mul__(self,scalar):
      newMatrix=[ [0 for i in range(len(self.data))] for j in range(len(self.data[0]))]
      for i in range(len(self.data)):
        for j in range(len(self.data[0])):
          newMatrix[i][j]=self.data[i][j]*scalar
      return newMatrix

    def shape(self):
      # matrix is a list containing 3 lists
      # matrix[0] is [1, 2, 3] (first row)
      # matrix[1] is [4, 5, 6] (second row) Each row is a list of column values
      rows=len(self.data) #returns a list of lists  ,number of rows (outer list length)
      cols=len(self.data[0]) #returns    number of columns (inner list length)
      return f'({rows},{cols})'

    @staticmethod
    def identity(n):
      newMatrix=[[1 if i==j else 0 for i in range(n) ]for j in range(n)]
      return f"Identity: {newMatrix}"

m1 = Matrix([[1, 2], [3, 4]])
m2 = Matrix([[5, 6], [7, 8]])

print(m1)               # prints rows
# print(m1.shape(m1))
print(m1.shape())         # (2, 2)
print(m1 + m2)          # [[6, 8], [10, 12]]
# print(m1.__add__(m2))
# print(m1.__mul__(3))
print(m1 * 3)           # [[3, 6], [9, 12]]
print(Matrix.identity(3))


1 2
3 4
(2,2)
[[6, 8], [10, 12]]
[[3, 6], [9, 12]]
Identity: [[1, 0, 0], [0, 1, 0], [0, 0, 1]]


### Q3

Build a shape hierarchy:

- `Shape` base class with a `@property` called `area` that raises `NotImplementedError`
- `Rectangle(Shape)` — takes `width` and `height`
- `Circle(Shape)` — takes `radius` (use `pi = 3.14159`)
- `Triangle(Shape)` — takes `base` and `height`
- All three override `area`
- Add `__gt__`, `__lt__`, `__eq__` to `Shape` so shapes can be compared and sorted by area
- Add `__str__` to each subclass showing its type and area rounded to 2 decimal places


In [ ]:

class Shape:

    def __str__(self):
      # return f'[shape]: {self.__class__.__name__} [area]: {self.area()}' without property
      return f'[shape]: {self.__class__.__name__} [area]: {self.area}'
    @property
    def area(self):
      raise NotImplementedError

    def __gt__(self,other):
      if not isinstance(other,Shape):
        return False
      return self.area>other.area
    def __lt__(self,other):
      if not isinstance(other,Shape):
        return False
      return self.area<other.area
    def __eq__(self,other):
      if not isinstance(other,Shape):
        return False
      return self.area==other.area

class Rectangle(Shape):
    ### Code here
    def __init__(self,width,height):
      self.width=width
      self.height=height
    @property
    def area(self):
      return self.width*self.height

class Circle(Shape):
    pi = 3.14159
    def __init__(self,radius):
      self.radius=radius
    @property
    def area(self):
      return Circle.pi*self.radius**2

class Triangle(Shape):
    def __init__(self,base,height):
      self.base=base
      self.height=height
    @property
    def area(self):
      return 0.5*self.base*self.height

shapes = [Rectangle(4, 5), Circle(3), Triangle(6, 8)]
print('Before Sorting')
for s in shapes:
    print(s)
print('After Sorting')
shapes.sort()   # uses __lt__
for s in shapes:
    print(s)


print(Rectangle(4, 5) > Circle(1))   # True


Before Sorting
[shape]: Rectangle [area]: 20
[shape]: Circle [area]: 28.27431
[shape]: Triangle [area]: 24.0
After Sorting
[shape]: Rectangle [area]: 20
[shape]: Triangle [area]: 24.0
[shape]: Circle [area]: 28.27431
True


---
## Section 2: Functional Python

### 2.1 map, filter, reduce

These three functions let you process collections in a clean, expressive style — without writing explicit `for` loops.

- `map(func, iterable)` — apply `func` to every element
- `filter(func, iterable)` — keep only elements where `func` returns `True`
- `reduce(func, iterable)` — combine all elements into a single value (needs `from functools import reduce`)


In [ ]:
from functools import reduce

numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# map: square every number
squares = list(map(lambda x: x**2, numbers))
print("Squares:", squares)
# [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]

# filter: keep only even numbers
evens = list(filter(lambda x: x % 2 == 0, numbers))
print("Evens:", evens)
# [2, 4, 6, 8, 10]

# reduce: sum all numbers
total = reduce(lambda acc, x: acc + x, numbers)
print("Total:", total)
# 55

# Chaining all three: square the evens, then sum them
result = reduce(lambda acc, x: acc + x,
                map(lambda x: x**2,
                    filter(lambda x: x % 2 == 0, numbers)))
print("Sum of squares of evens:", result)
# 4 + 16 + 36 + 64 + 100 = 220


Squares: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
Evens: [2, 4, 6, 8, 10]
Total: 55
Sum of squares of evens: 220


### 2.2 *args and **kwargs

- `*args` lets a function accept **any number of positional arguments** — they come in as a tuple
- `**kwargs` lets a function accept **any number of keyword arguments** — they come in as a dict

This is how functions like `print()` can take unlimited arguments.


In [ ]:
# *args — any number of positional arguments
def add_all(*args):
    print(f"args received: {args}")
    return sum(args)

print(add_all(1, 2, 3))         # 6
print(add_all(1, 2, 3, 4, 5))   # 15

print()

# **kwargs — any number of keyword arguments
def print_profile(**kwargs):
    for key, value in kwargs.items():
        print(f"  {key}: {value}")

print_profile(name="Ali", age=25, city="Lahore", role="Trainee")

print()

# Combining: fixed args + *args + **kwargs
def mixed(a, b, *args, **kwargs):
    print(f"Required: a={a}, b={b}")
    print(f"Extra positional: {args}")
    print(f"Extra keyword: {kwargs}")

mixed(1, 2, 3, 4, x=10, y=20)


args received: (1, 2, 3)
6
args received: (1, 2, 3, 4, 5)
15

  name: Ali
  age: 25
  city: Lahore
  role: Trainee

Required: a=1, b=2
Extra positional: (3, 4)
Extra keyword: {'x': 10, 'y': 20}


### 2.3 Decorators

A **decorator** is a function that **wraps another function** to add behaviour — without modifying the original function's code.

Think of it like a wrapper around a gift: the gift (your function) stays the same, but the wrapper (decorator) adds something extra — like timing, logging, or validation.

The `@decorator_name` syntax is just shorthand for `func = decorator(func)`.


In [ ]:
import time

# ── Basic decorator: measures how long a function takes ──
def timer(func):
    def wrapper(*args, **kwargs):         # *args/**kwargs so it works on ANY function
        start = time.time()
        result = func(*args, **kwargs)    # call the original function
        end = time.time()
        print(f"{func.__name__} took {end - start:.4f} seconds")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

slow_sum(1_000_000)   # slow_sum took ~0.03 seconds

print()

# ── Decorator with arguments: repeat a function N times ──
def repeat(n):
    def decorator(func):
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def greet(name):
    print(f"Hello, {name}!")

greet("Ali")    # prints "Hello, Ali!" 3 times


### 2.4 Generators

A **generator** produces values **one at a time** using the `yield` keyword. Unlike a list, it doesn't compute and store everything in memory upfront — it generates each value only when asked.

This makes generators extremely useful when working with **large datasets** in ML (streaming batches, lazy loading, etc.).


In [ ]:
# Regular function: builds the entire list in memory first
def squares_list(n):
    return [x**2 for x in range(n)]

# Generator: produces one value at a time — memory efficient
def squares_gen(n):
    for x in range(n):
        yield x**2      # pauses here; resumes on next call to next()

# Using next() manually
gen = squares_gen(5)
print(next(gen))   # 0
print(next(gen))   # 1
print(next(gen))   # 4

print()

# Or just loop over it
for val in squares_gen(5):
    print(val, end=" ")
print()

print()

# Generator expression (lazy list comprehension)
gen_expr = (x**2 for x in range(10))
print(list(gen_expr))   # [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]

# Practical example: batch generator for ML training
def batch_generator(data, batch_size):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

dataset = list(range(20))
for batch in batch_generator(dataset, 5):
    print("Batch:", batch)


0
1
4

0 1 4 9 16 

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
Batch: [0, 1, 2, 3, 4]
Batch: [5, 6, 7, 8, 9]
Batch: [10, 11, 12, 13, 14]
Batch: [15, 16, 17, 18, 19]


---
## Functional Python Tasks

### Q4

Write a function `pipeline(data, *funcs)` that takes a list and any number of functions, and applies them **in sequence** using `map` or `filter`. Return the final result as a list.

- If the function returns a boolean (i.e., it's a filter), use `filter`
- If it transforms values, use `map`

**Hint**: you can check a function's behavior by testing it on a sample value — if the result is `True`/`False`, it's a filter.


In [ ]:
def pipeline(data, *funcs):
    ## Code here

     newList=list(map(square,filter(is_even,map(double,data))))
     return newList

# def pipeline(data, *funcs):
#     result = list(data)
#     print(result)
#     for func in funcs:
#         # Apply each function to the data
#         result = [func(x) for x in result]
#         print(result)
#     return result

double   = lambda x: x * 2
is_even  = lambda x: x % 2 == 0
square   = lambda x: x ** 2

data = [1, 2, 3, 4, 5]
result = pipeline(data, double, is_even, square)
print(result)   # [16, 64]
# Step 1 (double):  [2, 4, 6, 8, 10]
# Step 2 (is_even): [2, 4, 6, 8, 10]  (all are even after doubling)
# Step 3 (square):  [4, 16, 36, 64, 100]
# Wait — re-read: filter BEFORE square: [2,4,6,8,10] -> all even -> square -> [4,16,36,64,100]


[4, 16, 36, 64, 100]


### Q5

Write a decorator `@validate_positive` that raises a `ValueError` if **any argument** passed to the decorated function is negative or zero.

The decorator should work on functions with any number of arguments.


In [ ]:
def validate_positive(func):
    ### Code here
    def wrapper(*args,**kwargs):

      # Check all keyword arguments
      for element in args:
       if element <0:
        raise ValueError(f'Argument {args} cannot be negative')

       # Check all keyword arguments
      for key, value in kwargs.items():
       if value <= 0:
        raise ValueError(f"Argument {key}={value} must be positive")


      result=func(*args,**kwargs)
      return result
    return wrapper


@validate_positive
def multiply(a, b):
    return a * b

@validate_positive
def add_three(a, b, c):
    return a + b + c

print(multiply(3, 4))       # 12
print(add_three(1, 2, 3))   # 6

try:
    multiply(-1, 4)         # Should raise ValueError
except ValueError as e:
    print(e)

try:
    multiply(3, 0)          # Should raise ValueError
except ValueError as e:
    print(e)


12
6
Argument (-1, 4) cannot be negative


### Q6

Write a **generator function** `running_stats(numbers)` that yields a tuple `(mean, variance)` after each new number is added to the stream.

**Important constraint**: do NOT store the full list or recompute from scratch each time. Update mean and variance **incrementally** using Welford's online algorithm or a running sum approach.

This is exactly how streaming ML systems compute statistics on live data.


In [ ]:
def running_stats(numbers):

  n=0
  mean=0.0      # Running mean
  M2=0.0        # Running sum of squared differences from the mean

  for x in numbers:
        n += 1
        delta = x - mean
        mean = mean + delta / n
        # delta = x - old_mean (difference from old mean)
        # (x - mean) = difference from new mean
        M2 = M2 + delta * (x - mean)
        variance = M2 / n  # population variance
        yield mean, variance

#  for n in range(len(numbers)):
#     mean=sum(numbers[:n+1])/len(numbers[:n+1])
#     sum_sq_diff =sum((numbers[n]-mean)**2 for i in range(n+1))
#     variance=sum_sq_diff/(len(numbers)-1)
#     yield (mean,variance)

for mean, var in running_stats([2, 4, 4, 4, 5, 5, 7, 9]):
    print(f"Mean: {mean:.2f}, Variance: {var:.2f}")

# Expected output:
# Mean: 2.00, Variance: 0.00
# Mean: 3.00, Variance: 1.00
# Mean: 3.33, Variance: 0.89
# Mean: 3.50, Variance: 0.75
# Mean: 3.80, Variance: 1.36
# ...


Mean: 2.00, Variance: 0.00
Mean: 3.00, Variance: 1.00
Mean: 3.33, Variance: 0.89
Mean: 3.50, Variance: 0.75
Mean: 3.80, Variance: 0.96
Mean: 4.00, Variance: 1.00
Mean: 4.43, Variance: 1.96
Mean: 5.00, Variance: 4.00


### Q7

Using **only** `map`, `filter`, and `reduce` (no `for` or `while` loops), do the following in one pipeline:

1. Take the list of sentences below
2. Filter sentences that have **more than 5 words**
3. Convert each to **title case**
4. Concatenate all into one string separated by `" | "`


In [ ]:
from functools import reduce

sentences = [
    "the quick brown fox jumps over the lazy dog",
    "hello world",
    "artificial intelligence is transforming the world today",
    "python",
    "machine learning requires a lot of data and compute"
]

# result1 =  (' | ').join(map(lambda x:x.title(),(filter(lambda x :len(x.split())>5,sentences))))
result1=reduce(lambda x,acc:x+ ' | '+acc,(map(lambda x:x.title(),(filter(lambda x :len(x.split())>5,sentences)))))
result=(result1)
#  [sentence for sentence in sentences]

print(result)
# Expected:
# "The Quick Brown Fox Jumps Over The Lazy Dog | Artificial Intelligence Is Transforming The World Today | Machine Learning Requires A Lot Of Data And Compute"
# join() is  doing the same thing as reduce - it concatenates all elements with a separator.    #" | ".join( map(...)

The Quick Brown Fox Jumps Over The Lazy Dog | Artificial Intelligence Is Transforming The World Today | Machine Learning Requires A Lot Of Data And Compute


---
## Section 3: NumPy

### 3.1 Creating Arrays

NumPy is the backbone of almost every ML library (TensorFlow, PyTorch, scikit-learn all use NumPy arrays internally). Its core object is the `ndarray` — an n-dimensional array that supports fast, vectorized operations.

Key advantage over Python lists: NumPy operations run in **compiled C code**, making them 10–100x faster than equivalent Python loops.


In [ ]:
import numpy as np

# From a Python list
a = np.array([1, 2, 3, 4, 5])
print("1D array:", a)
print("dtype:", a.dtype)      # int64
print("shape:", a.shape)      # (5,)

print()

# 2D array (matrix) — rows x columns
b = np.array([[1, 2, 3],
              [4, 5, 6]])
print("2D array shape:", b.shape)   # (2, 3)

print()

# Useful constructors
print("zeros:\n",   np.zeros((3, 3)))
print("ones:\n",    np.ones((2, 4)))
print("identity:\n", np.eye(3))
print("arange:", np.arange(0, 10, 2))        # [0 2 4 6 8]
print("linspace:", np.linspace(0, 1, 5))     # [0. 0.25 0.5 0.75 1.]

np.random.seed(42)
print("random normal:\n", np.random.randn(3, 3).round(2))


1D array: [1 2 3 4 5]
dtype: int64
shape: (5,)

2D array shape: (2, 3)

zeros:
 [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
ones:
 [[1. 1. 1. 1.]
 [1. 1. 1. 1.]]
identity:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
arange: [0 2 4 6 8]
linspace: [0.   0.25 0.5  0.75 1.  ]
random normal:
 [[ 0.49  -0.14  0.65]
 [ 1.52  1.46 -0.23]
 [-0.23 -0.23 -0.46]]


### 3.2 Indexing and Slicing

NumPy indexing uses `[row, col]` notation for 2D arrays. You can also use **boolean masks** to select elements based on a condition — this is used constantly in data preprocessing.


In [ ]:
import numpy as np

a = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

print("Element [0,1]:", a[0, 1])       # 2  — row 0, col 1
print("Column 1:", a[:, 1])            # [2 5 8] — all rows, col 1
print("Row 1:", a[1, :])               # [4 5 6] — row 1, all cols
print("Submatrix:\n", a[0:2, 0:2])   # [[1 2], [4 5]]

print()

# Boolean masking — returns elements where condition is True
print("Elements > 5:", a[a > 5])       # [6 7 8 9]

# Set all even elements to 0
b = a.copy()
b[b % 2 == 0] = 0
print("Even elements zeroed:\n", b)

print()

# Fancy indexing — select specific rows
print("Rows 0 and 2:\n", a[[0, 2]])


Element [0,1]: 2
Column 1: [2 5 8]
Row 1: [4 5 6]
Submatrix:
 [[1 2]
 [4 5]]

Elements > 5: [6 7 8 9]
Even elements zeroed:
 [[1 0 3]
 [0 5 0]
 [7 0 9]]

Rows 0 and 2:
 [[1 2 3]
 [7 8 9]]


### 3.3 Array Math and Broadcasting

NumPy operations are **element-wise by default** — no loops needed.

**Broadcasting** is NumPy's way of performing operations between arrays of different shapes. The smaller array is automatically "stretched" to match the larger one. This is heavily used in ML — for example, subtracting the mean from each row of a dataset.


In [ ]:
import numpy as np

x = np.array([1, 2, 3])
y = np.array([4, 5, 6])

# Element-wise operations
print("x + y =", x + y)        # [5 7 9]
print("x * y =", x * y)        # [4 10 18]
print("x ** 2 =", x ** 2)      # [1 4 9]

# Dot product
print("dot(x,y) =", np.dot(x, y))   # 1*4 + 2*5 + 3*6 = 32

print()

# Matrix multiplication
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print("A @ B =\n", A @ B)     # [[19 22], [43 50]]

print()

# Broadcasting: add a 1D array to each row of a 2D array
data = np.array([[1, 2, 3],
                 [4, 5, 6]])        # shape (2, 3)
bias = np.array([10, 20, 30])       # shape (3,) — broadcast across rows

print("data + bias =\n", data + bias)   # [[11 22 33], [14 25 36]]

# Broadcasting: subtract column mean from each column (normalization)
col_means = data.mean(axis=0)            # shape (3,) — mean of each column
print("column means:", col_means)
print("centered:\n", data - col_means)


x + y = [5 7 9]
x * y = [ 4 10 18]
x ** 2 = [1 4 9]
dot(x,y) = 32

A @ B =
 [[19 22]
 [43 50]]

data + bias =
 [[11 22 33]
 [14 25 36]]
column means: [2.5 3.5 4.5]
centered:
 [[-1.5 -1.5 -1.5]
 [ 1.5  1.5  1.5]]


### 3.4 Useful Aggregate Operations

These operations are used constantly in ML — computing loss functions, accuracy, normalization, etc.


In [ ]:
import numpy as np

a = np.array([[1, 2, 3],
              [4, 5, 6]])

print("Sum (all):", a.sum())           # 21
print("Sum (per col):", a.sum(axis=0)) # [5 7 9]
print("Sum (per row):", a.sum(axis=1)) # [6 15]
print("Mean:", a.mean())               # 3.5
print("Std:", a.std().round(4))        # 1.7078
print("Max:", a.max())                 # 6
print("Min:", a.min())                 # 1
print("Argmax:", a.argmax())           # 5 (flat index of max element)
print("Argmax per row:", a.argmax(axis=1))  # [2 2] — col index of max in each row

print()
print("Reshape to (3,2):\n", a.reshape(3, 2))
print("Transpose:\n", a.T)
print("Flatten:", a.flatten())


Sum (all): 21
Sum (per col): [5 7 9]
Sum (per row): [ 6 15]
Mean: 3.5
Std: 1.7078
Max: 6
Min: 1
Argmax: 5
Argmax per row: [2 2]

Reshape to (3,2):
 [[1 2]
 [3 4]
 [5 6]]
Transpose:
 [[1 4]
 [2 5]
 [3 6]]
Flatten: [1 2 3 4 5 6]


---
## NumPy Tasks

### Q8

Given the 2D NumPy array `scores` below (rows = students, columns = subjects), write NumPy code (no loops) to:

1. Find the **mean score per student** (one value per student)
2. Find the **index of the top scoring student** (highest mean)
3. **Normalize** all scores to range [0, 1] using min-max normalization: `(x - min) / (max - min)`
4. Count how many students scored **above 70 in every subject**


In [ ]:
import numpy as np

scores = np.array([
    [85, 92, 78, 90],
    [60, 55, 70, 65],
    [95, 88, 92, 97],
    [72, 68, 80, 74],
    [40, 55, 60, 50]
])

# 1. Mean score per student
### Code here
mean=scores.mean(axis=0)
print(mean)

# 2. Index of top scoring student
### Code here
# top_score=scores.argmax()
# top_score=max(mean)
top_score=mean.argmax()
print(top_score)

# 3. Normalize scores to [0, 1]
### Code here
# normalize=(x - min) / (max - min)
# score=scores.flatten()
# print(score)
normalize= (scores-scores.min())/ (scores.max()-scores.min())
print(normalize)

# 4. Students who scored above 70 in ALL subjects
### Code here
stds_who_scored=(scores>70).all(axis=1)
print(stds_who_scored.sum())

print(f"Students (indices) who scored above 70 in all subjects: {np.where(stds_who_scored)[0]}")
# mean = scores.mean(axis=0)  #  This gives mean per SUBJECT (column)
# mean = scores.mean(axis=1)  #  One value per student


[70.4 71.6 76.  75.2]
2
[[0.78947368 0.9122807  0.66666667 0.87719298]
 [0.35087719 0.26315789 0.52631579 0.43859649]
 [0.96491228 0.84210526 0.9122807  1.        ]
 [0.56140351 0.49122807 0.70175439 0.59649123]
 [0.         0.26315789 0.35087719 0.1754386 ]]
2
Students (indices) who scored above 70 in all subjects: [0 2]


### Q9

Implement the following three functions using **only NumPy** — no Python loops:

1. `cosine_similarity(a, b)` — cosine similarity between two 1D vectors: `dot(a,b) / (||a|| * ||b||)`
2. `softmax(x)` — converts a 1D array to probabilities that sum to 1: `exp(x) / sum(exp(x))`
3. `batch_normalize(X)` — normalize each **column** of a 2D array to have mean=0, std=1

These three functions are used directly in neural networks and ML pipelines.


In [ ]:
import numpy as np

def cosine_similarity(a, b):
    ### Code here

    dot_product = np.dot(a, b)
    # L2 norms (magnitudes) of each vector
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    # Cosine similarity
    return dot_product / (norm_a * norm_b)


def softmax(x):
    ### Code here
    # exp(x) / sum(exp(x))

     # Subtract max for numerical stability (prevents overflow)
    x_shifted = x - np.max(x)

    # Compute exponentials
    exp_x = np.exp(x_shifted)

    # Sum of exponentials
    sum_exp_x = np.sum(exp_x)

    # Return probabilities
    return exp_x / sum_exp_x


def batch_normalize(X):
    ### Code here
    # normalize each column of a 2D array to have mean=0, std=1,  #(X - mean) / std for each column

    # Compute mean of each column (axis=0 means column-wise)
    mean = X.mean(axis=0, keepdims=True)

    # Compute standard deviation of each column
    std = X.std(axis=0, keepdims=True)

    # Normalize: (X - mean) / std
    # Add small epsilon to avoid division by zero
    epsilon = 1e-10
    return (X - mean) / (std + epsilon)

# keepdims=0 is the same as keepdims=False.
# keepdims determines whether the reduced dimensions are kept as dimensions of size 1 or removed.

# When to Use Which
# Use keepdims=True when:
# You need to broadcast with the original array
# You want to maintain the same number of dimensions
# You're doing operations like (X - mean) / std

# Use keepdims=False when:
# You want a flat 1D result
# You're just extracting statistics (like for printing)
# You don't need to broadcast
# dot product/product of magnitudes of 2 vectors

# Test cosine_similarity
a = np.array([1, 2, 3], dtype=float)
b = np.array([4, 5, 6], dtype=float)
print("Cosine similarity:", cosine_similarity(a, b))   # ~0.9746

# Test softmax
x = np.array([1.0, 2.0, 3.0])
probs = softmax(x)
print("Softmax:", probs.round(4))      # [0.0900 0.2447 0.6652]
print("Sum:", probs.sum())              # 1.0

# Test batch_normalize
X = np.array([[1, 10],
              [2, 20],
              [3, 30]], dtype=float)
normed = batch_normalize(X)
print("Normalized:\n", normed.round(4))
print("Column means:", normed.mean(axis=0).round(10))   # [~0. ~0.]
print("Column stds:", normed.std(axis=0).round(10))     # [~1. ~1.]


Cosine similarity: 0.9746318461970762
Softmax: [0.09   0.2447 0.6652]
Sum: 0.9999999999999999
Normalized:
 [[-1.2247 -1.2247]
 [ 0.      0.    ]
 [ 1.2247  1.2247]]
Column means: [0. 0.]
Column stds: [1. 1.]
